# 03 - Train MPNN-POM

Step 6 of the guide: the `mpnn_pom_train.ipynb` notebook from the OpenPOM repo with the file paths pointed at the local CSV.

Run it once with 5 epochs to confirm the pipeline works, then raise `nb_epoch` (try 20, then 50; the paper used 150).

In [1]:
import deepchem as dc
from openpom.feat.graph_featurizer import GraphFeaturizer, GraphConvConstants
from openpom.utils.data_utils import get_class_imbalance_ratio, IterativeStratifiedSplitter
from openpom.models.mpnn_pom import MPNNPOMModel
from datetime import datetime

No normalization for SPS. Feature removed!


No normalization for AvgIpc. Feature removed!


No normalization for NumAmideBonds. Feature removed!


No normalization for NumAtomStereoCenters. Feature removed!


No normalization for NumBridgeheadAtoms. Feature removed!


No normalization for NumHeterocycles. Feature removed!


... (7 outputs left out)

Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'transformers'


Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'


Skipped loading some Jax models, missing a dependency. No module named 'jax'


Skipped loading some PyTorch models, missing a dependency. No module named 'tensorflow'


In [2]:
TASKS = [
'alcoholic', 'aldehydic', 'alliaceous', 'almond', 'amber', 'animal',
'anisic', 'apple', 'apricot', 'aromatic', 'balsamic', 'banana', 'beefy',
'bergamot', 'berry', 'bitter', 'black currant', 'brandy', 'burnt',
'buttery', 'cabbage', 'camphoreous', 'caramellic', 'cedar', 'celery',
'chamomile', 'cheesy', 'cherry', 'chocolate', 'cinnamon', 'citrus', 'clean',
'clove', 'cocoa', 'coconut', 'coffee', 'cognac', 'cooked', 'cooling',
'cortex', 'coumarinic', 'creamy', 'cucumber', 'dairy', 'dry', 'earthy',
'ethereal', 'fatty', 'fermented', 'fishy', 'floral', 'fresh', 'fruit skin',
'fruity', 'garlic', 'gassy', 'geranium', 'grape', 'grapefruit', 'grassy',
'green', 'hawthorn', 'hay', 'hazelnut', 'herbal', 'honey', 'hyacinth',
'jasmin', 'juicy', 'ketonic', 'lactonic', 'lavender', 'leafy', 'leathery',
'lemon', 'lily', 'malty', 'meaty', 'medicinal', 'melon', 'metallic',
'milky', 'mint', 'muguet', 'mushroom', 'musk', 'musty', 'natural', 'nutty',
'odorless', 'oily', 'onion', 'orange', 'orangeflower', 'orris', 'ozone',
'peach', 'pear', 'phenolic', 'pine', 'pineapple', 'plum', 'popcorn',
'potato', 'powdery', 'pungent', 'radish', 'raspberry', 'ripe', 'roasted',
'rose', 'rummy', 'sandalwood', 'savory', 'sharp', 'smoky', 'soapy',
'solvent', 'sour', 'spicy', 'strawberry', 'sulfurous', 'sweaty', 'sweet',
'tea', 'terpenic', 'tobacco', 'tomato', 'tropical', 'vanilla', 'vegetable',
'vetiver', 'violet', 'warm', 'waxy', 'weedy', 'winey', 'woody'
]
print("No of tasks: ", len(TASKS))

No of tasks:  138


Dataset path is fixed to the clone in `openpom/`, so no download is needed.

In [3]:
input_file = "openpom/openpom/data/curated_datasets/curated_GS_LF_merged_4983.csv"

featurizer = GraphFeaturizer()
smiles_field = "nonStereoSMILES"
loader = dc.data.CSVLoader(tasks=TASKS, feature_field=smiles_field, featurizer=featurizer)
dataset = loader.create_dataset(inputs=[input_file])
n_tasks = len(dataset.tasks)
print("n_tasks:", n_tasks)
print("n_molecules:", len(dataset))

n_tasks: 138
n_molecules: 4983


In [4]:
# check total datapoints per odor
import pandas as pd

df = pd.DataFrame([{'odor': TASKS[i], 'count': dataset.y[:, i].sum()} for i in range(138)])
df.sort_values(by='count')

,odor,count
25,chamomile,31.0
13,bergamot,32.0
76,malty,33.0
93,orangeflower,33.0
61,hawthorn,34.0
...,...,...
64,herbal,756.0
50,floral,1100.0
123,sweet,1429.0
60,green,1446.0


## Split 80 / 10 / 10 (paper version, multi-label stratified)

In [5]:
splitter = IterativeStratifiedSplitter(order=2)
train_dataset, valid_dataset, test_dataset = splitter.train_valid_test_split(dataset, frac_train=0.8, frac_valid=0.1, frac_test=0.1)

In [6]:
# check for data leakage
assert len(set(train_dataset.ids).intersection(set(valid_dataset.ids))) == 0
assert len(set(train_dataset.ids).intersection(set(test_dataset.ids))) == 0
assert len(set(valid_dataset.ids).intersection(set(test_dataset.ids))) == 0

print("train_dataset: ", len(train_dataset))
print("valid_dataset: ", len(valid_dataset))
print("test_dataset : ", len(test_dataset))

train_dataset:  3978
valid_dataset:  493
test_dataset :  512


In [7]:
train_ratios = get_class_imbalance_ratio(train_dataset)
assert len(train_ratios) == n_tasks
print("class imbalance ratios ready:", len(train_ratios))

class imbalance ratios ready: 138


In [8]:
learning_rate = dc.models.optimizers.ExponentialDecay(initial_rate=0.001, decay_rate=0.5, decay_steps=32*15, staircase=True)

In [9]:
import torch

model = MPNNPOMModel(n_tasks = n_tasks,
                        batch_size=128,
                        learning_rate=learning_rate,
                        class_imbalance_ratio = train_ratios,
                        loss_aggr_type = 'sum',
                        node_out_feats = 100,
                        edge_hidden_feats = 75,
                        edge_out_feats = 100,
                        num_step_message_passing = 5,
                        mpnn_residual = True,
                        message_aggregator_type = 'sum',
                        mode = 'classification',
                        number_atom_features = GraphConvConstants.ATOM_FDIM,
                        number_bond_features = GraphConvConstants.BOND_FDIM,
                        n_classes = 1,
                        readout_type = 'set2set',
                        num_step_set2set = 3,
                        num_layer_set2set = 2,
                        ffn_hidden_list= [392, 392],
                        ffn_embeddings = 256,
                        ffn_activation = 'relu',
                        ffn_dropout_p = 0.12,
                        ffn_dropout_at_input_no_act = False,
                        weight_decay = 1e-5,
                        self_loop = False,
                        optimizer_name = 'adam',
                        log_frequency = 32,
                        model_dir = './models/03_train',
                        device_name = 'cuda' if torch.cuda.is_available() else 'cpu')

### Epochs

Start with 5 to check everything runs. After that works, raise it: 20, then 50 (paper: 150).

In [10]:
nb_epoch = 150

In [11]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)

In [12]:
start_time = datetime.now()
for epoch in range(1, nb_epoch+1):
        loss = model.fit(
              train_dataset,
              nb_epoch=1,
              max_checkpoints_to_keep=1,
              deterministic=False,
              restore=epoch>1)
        train_scores = model.evaluate(train_dataset, [metric])['roc_auc_score']
        valid_scores = model.evaluate(valid_dataset, [metric])['roc_auc_score']
        print(f"epoch {epoch}/{nb_epoch} ; loss = {loss}; train_scores = {train_scores}; valid_scores = {valid_scores}")
model.save_checkpoint()
end_time = datetime.now()

epoch 1/150 ; loss = 3.8573527336120605; train_scores = 0.642317705617149; valid_scores = 0.642857150896672


/home/moosa/Downloads/smell-agent/.venv/lib/python3.10/site-packages/deepchem/models/torch_models/torch_model.py:1087: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = t

epoch 2/150 ; loss = 3.171921491622925; train_scores = 0.750471233926666; valid_scores = 0.7241592320464832


/home/moosa/Downloads/smell-agent/.venv/lib/python3.10/site-packages/deepchem/models/torch_models/torch_model.py:1087: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = t

epoch 3/150 ; loss = 3.004920482635498; train_scores = 0.7747548985153405; valid_scores = 0.7557348911174849


/home/moosa/Downloads/smell-agent/.venv/lib/python3.10/site-packages/deepchem/models/torch_models/torch_model.py:1087: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = t

... (289 outputs left out)

/home/moosa/Downloads/smell-agent/.venv/lib/python3.10/site-packages/deepchem/models/torch_models/torch_model.py:1087: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = t

epoch 149/150 ; loss = 1.5277431011199951; train_scores = 0.9556783372987078; valid_scores = 0.8795346192963158


/home/moosa/Downloads/smell-agent/.venv/lib/python3.10/site-packages/deepchem/models/torch_models/torch_model.py:1087: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = t

epoch 150/150 ; loss = 1.5236363410949707; train_scores = 0.955985630640989; valid_scores = 0.8796289806202212


## Step 7: read the score

ROC-AUC on the test set, averaged over all 138 words. 0.5 = guessing, 1.0 = perfect. A full 10-model ensemble reaches about 0.89; one short run lands lower, which is normal.

In [13]:
test_scores = model.evaluate(test_dataset, [metric])['roc_auc_score']
print("time_taken: ", str(end_time-start_time))
print("test_score: ", test_scores)

time_taken:  0:12:30.244451
test_score:  0.8752930828496587


In [14]:
import pathlib

out = pathlib.Path('./models/03_train')
out.mkdir(parents=True, exist_ok=True)
(out / 'test_score.txt').write_text(
    f"epochs={nb_epoch}\ntest_roc_auc={test_scores}\ntime={end_time-start_time}\n")
print((out / 'test_score.txt').read_text())

epochs=150
test_roc_auc=0.8752930828496587
time=0:12:30.244451

